# BadBlueprint full scoring (Colab)

This notebook runs the BadBlueprint **full scoring** flow using:
- **Model constraint**: `openai/gpt-oss-20b` (open-weight; no proprietary API advantage)
- A **local OpenAI-compatible endpoint** started inside Colab via `transformers serve`
- The **upstream** `agentbeats-lambda` harness (forked) cloned into `/content/agentbeats-lambda`

Outputs are normalized to `results/badblueprint/*` and packaged for download.


## A. Runtime / GPU check


In [5]:
import platform
import shutil
import subprocess

print("Python:", platform.python_version())

def run(cmd: str) -> int:
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, check=False)
    return p.returncode

# GPU info (best-effort)
run("nvidia-smi || true")

# Torch CUDA check (Colab usually has torch preinstalled)
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("WARNING: torch check failed:", repr(e))

# System memory / disk
run("free -h || true")
run("df -h / || true")

if shutil.which("nvidia-smi") is None:
    print("WARNING: No GPU detected. Full scoring may be extremely slow or fail.")



Python: 3.10.12
$ nvidia-smi || true
Wed Jan 14 22:51:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10                     On  |   00000000:07:00.0 Off |                    0 |
|  0%   28C    P8             11W /  150W |       3MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------

## B. Environment setup


In [16]:
import os, sys, subprocess
from pathlib import Path

def sh(cmd: str):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

ROOT = Path("/content")
ROOT.mkdir(parents=True, exist_ok=True)

VENV = ROOT / "llm-venv"
if not VENV.exists():
    sh(f"python3 -m venv {VENV}")

PIP = VENV / "bin" / "pip"
PY  = VENV / "bin" / "python"

# Always upgrade pip tooling inside venv
sh(f"{PIP} install -U pip setuptools wheel")

# Core deps (pin hf hub per your earlier requirement)
sh(f'{PIP} install -U --quiet numpy "pillow>=10.0.0" "transformers[serving]" "huggingface-hub>=0.34.0,<1.0"')
sh(f'{PIP} install -U --quiet vllm')
sh(f"{PIP} install -U --quiet accelerate safetensors requests hf_transfer || true")

# Make THIS notebook process import from the venv (so `import transformers` uses venv packages)
site_pkgs = next(VENV.glob("lib/python*/site-packages"))
sys.path.insert(0, str(site_pkgs))
os.environ["VIRTUAL_ENV"] = str(VENV)
os.environ["PATH"] = str(VENV / "bin") + ":" + os.environ.get("PATH", "")

sh(f'{PY} -c "import vllm, transformers, huggingface_hub, PIL, numpy; import PIL.Image as I; '
   'print(\'vllm\', vllm.__version__); '
   'print(\'transformers\', transformers.__version__); '
   'print(\'huggingface_hub\', huggingface_hub.__version__); '
   'print(\'pillow\', PIL.__version__, \'Resampling=\', hasattr(I, \'Resampling\')); '
   'print(\'numpy\', numpy.__version__)"')



$ /content/llm-venv/bin/pip install -U pip setuptools wheel
$ /content/llm-venv/bin/pip install -U --quiet numpy "pillow>=10.0.0" "transformers[serving]" "huggingface-hub>=0.34.0,<1.0"
$ /content/llm-venv/bin/pip install -U --quiet vllm
$ /content/llm-venv/bin/pip install -U --quiet accelerate safetensors requests hf_transfer || true
$ /content/llm-venv/bin/python -c "import vllm, transformers, huggingface_hub, PIL, numpy; import PIL.Image as I; print('vllm', vllm.__version__); print('transformers', transformers.__version__); print('huggingface_hub', huggingface_hub.__version__); print('pillow', PIL.__version__, 'Resampling=', hasattr(I, 'Resampling')); print('numpy', numpy.__version__)"
vllm 0.13.0
transformers 4.57.5
huggingface_hub 0.36.0
pillow 12.1.0 Resampling= True
numpy 2.2.6


## C. Model download


In [8]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os

MODEL_ID = "openai/gpt-oss-20b"
LOCAL_DIR = Path("/content/models/gpt-oss-20b")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set HF_TOKEN in Colab Secrets or environment (do not print it)
hf_token = os.environ.get("HF_TOKEN")

print("Downloading model (may take a while)...")
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(LOCAL_DIR),
    token=hf_token,
)

file_count = sum(1 for _ in LOCAL_DIR.rglob("*"))
print(f"Model downloaded to: {LOCAL_DIR} (files: {file_count})")



/content/llm-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 18 files: 100%|██████████| 18/18 [00:24<00:00,  1.34s/it]

Model downloaded to: /content/models/gpt-oss-20b (files: 62)


## D. Start local OpenAI-compatible endpoint


In [19]:
# Optional: kill whatever is currently listening on 8000
subprocess.run("sudo lsof -t -iTCP:8000 -sTCP:LISTEN | xargs -r sudo kill -9", shell=True)

CompletedProcess(args='sudo lsof -t -iTCP:8000 -sTCP:LISTEN | xargs -r sudo kill -9', returncode=0)

In [22]:
import os
import signal
import subprocess
import time
from pathlib import Path

MODEL_ID = "openai/gpt-oss-20b"
LOCAL_DIR = Path("/content/models/gpt-oss-20b")  # snapshot_download target
ENDPOINT = "http://127.0.0.1:8000/v1"

SERVER_LOG = Path("/content/vllm_server.log")
SERVER_PID = Path("/content/vllm_server.pid")

def is_pid_running(p: int) -> bool:
    try:
        os.kill(p, 0)
        return True
    except ProcessLookupError:
        return False

def stop_server():
    if not SERVER_PID.exists():
        return
    pid = int(SERVER_PID.read_text().strip())
    try:
        os.killpg(pid, signal.SIGTERM)
        time.sleep(2)
        if is_pid_running(pid):
            os.killpg(pid, signal.SIGKILL)
        print(f"Stopped server process group {pid}")
    except ProcessLookupError:
        print("Server process not running.")
    finally:
        SERVER_PID.unlink(missing_ok=True)

def start_server():
    # Sanity: ensure the local model dir exists and looks non-empty
    if not LOCAL_DIR.exists():
        raise FileNotFoundError(f"LOCAL_DIR does not exist: {LOCAL_DIR}")
    if not any(LOCAL_DIR.iterdir()):
        raise RuntimeError(f"LOCAL_DIR is empty (snapshot_download may have failed): {LOCAL_DIR}")

    # Remove stale PID
    if SERVER_PID.exists():
        pid = int(SERVER_PID.read_text().strip())
        if not is_pid_running(pid):
            print("Found stale PID file; removing.")
            SERVER_PID.unlink(missing_ok=True)

    if SERVER_PID.exists():
        print("Server appears to be running already. Skipping start.")
        return

    VENV = Path("/content/llm-venv")
    PY = str(VENV / "bin" / "python")

    # Start vLLM OpenAI-compatible server from LOCAL_DIR, but expose served name as MODEL_ID
    cmd = [
        PY, "-m", "vllm.entrypoints.openai.api_server",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--model", str(LOCAL_DIR),
        "--served-model-name", MODEL_ID,
    ]

    print("Starting vLLM server:", " ".join(cmd))
    with SERVER_LOG.open("w") as log_f:
        proc = subprocess.Popen(
            cmd,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            preexec_fn=os.setsid,  # process group for killpg
        )

    SERVER_PID.write_text(str(proc.pid))
    time.sleep(5)
    print(f"Server PID: {proc.pid}")
    print(f"Server log: {SERVER_LOG}")

start_server()
print("Model endpoint:", ENDPOINT)


Server appears to be running already. Skipping start.
Model endpoint: http://127.0.0.1:8000/v1


### Healthcheck


In [48]:
import time
import requests

ENDPOINT = "http://127.0.0.1:8000/v1"

ok = False
last_err = None
for _ in range(20):
    try:
        resp = requests.get(f"{ENDPOINT}/models", timeout=5)
        if resp.status_code == 200:
            print("Server is healthy.")
            data = resp.json()
            print(data)
            ids = {m.get("id") for m in data.get("data", [])}
            if MODEL_ID not in ids:
                raise RuntimeError(f"8000 is up but not serving {MODEL_ID}. Got: {sorted(list(ids))[:10]} ...")
            ok = True
            break
        last_err = f"HTTP {resp.status_code}: {resp.text[:200]}"
    except Exception as exc:
        last_err = repr(exc)
    time.sleep(3)

if not ok:
    raise RuntimeError(f"Server healthcheck failed: {last_err}")



Server is healthy.
{'object': 'list', 'data': [{'id': 'openai/gpt-oss-20b', 'object': 'model', 'created': 1768479985, 'owned_by': 'vllm', 'root': '/content/models/gpt-oss-20b', 'parent': None, 'max_model_len': 131072, 'permission': [{'id': 'modelperm-9359a5888121a851', 'object': 'model_permission', 'created': 1768479985, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


## E. Clone repo and prepare submission bundle


In [11]:
import subprocess
from pathlib import Path

def run(cmd: str, cwd: Path | None = None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=str(cwd) if cwd else None)

harness_dir = Path("/content/agentbeats-lambda")
if not harness_dir.exists():
    run("git clone https://github.com/purplevanguard/agentbeats-lambda.git", cwd=Path("/content"))

if (harness_dir / ".git").exists():
    run("git fetch --prune origin", cwd=harness_dir)
    run("git checkout -B submission/purple_vanguard_LegalEagleEye origin/submission/purple_vanguard_LegalEagleEye", cwd=harness_dir)
    run("git reset --hard origin/submission/purple_vanguard_LegalEagleEye", cwd=harness_dir)

print("Harness dir:", harness_dir)


$ git fetch --prune origin
$ git checkout -B submission/purple_vanguard_LegalEagleEye origin/submission/purple_vanguard_LegalEagleEye
Branch 'submission/purple_vanguard_LegalEagleEye' set up to track remote branch 'submission/purple_vanguard_LegalEagleEye' from 'origin'.
Your branch is up to date with 'origin/submission/purple_vanguard_LegalEagleEye'.
$ git reset --hard origin/submission/purple_vanguard_LegalEagleEye
HEAD is now at 649d5b7 Update plugin.py
Harness dir: /content/agentbeats-lambda


From https://github.com/purplevanguard/agentbeats-lambda
   b23afac..649d5b7  submission/purple_vanguard_LegalEagleEye -> origin/submission/purple_vanguard_LegalEagleEye
Reset branch 'submission/purple_vanguard_LegalEagleEye'


No manual copy needed. This notebook assumes the submission bundle is already present in the harness branch `submission/purple_vanguard` at:
`/content/agentbeats-lambda/scenarios/security_arena/submissions/purple_vanguard/badblueprint/`


In [7]:
from pathlib import Path

bundle_dir = Path("/content/agentbeats-lambda/scenarios/security_arena/submissions/purple_vanguard/badblueprint")
toml_path = bundle_dir / "scenario_badblueprint.toml"
plugin_path = bundle_dir / "plugin.py"
artifacts_dir = bundle_dir / "artifacts"
prompts_dir = bundle_dir / "prompts"

missing = []
if not toml_path.exists():
    missing.append(str(toml_path))
if not plugin_path.exists():
    missing.append(str(plugin_path))
if not artifacts_dir.is_dir() or not any(artifacts_dir.rglob("*")):
    missing.append(str(artifacts_dir))
if not prompts_dir.is_dir() or not any(prompts_dir.rglob("*")):
    missing.append(str(prompts_dir))

if missing:
    raise RuntimeError(
        "Missing required files in the harness branch submission/purple_vanguard. Sync the submission bundle to agentbeats-lambda branch `submission/purple_vanguard`."
    )

print("Submission bundle looks OK at", bundle_dir)


RuntimeError: Missing required files in the harness branch submission/purple_vanguard. Sync the submission bundle to agentbeats-lambda branch `submission/purple_vanguard`.

In [35]:
from pathlib import Path

harness_dir = Path("/content/agentbeats-lambda")
toml_path = harness_dir / "scenarios/security_arena/submissions/purple_vanguard/badblueprint/scenario_badblueprint.toml"

if not toml_path.exists():
    raise FileNotFoundError(f"Missing scenario toml: {toml_path}")

toml_text = toml_path.read_text(encoding="utf-8", errors="ignore")
bad_tokens = []
if "vendor/agentbeats-lambda/" in toml_text:
    bad_tokens.append("vendor/agentbeats-lambda/")
if "scripts/run_badblueprint_orchestrator.py" in toml_text:
    bad_tokens.append("scripts/run_badblueprint_orchestrator.py")
if bad_tokens:
    raise RuntimeError(
        "TOML still contains old paths: " + ", ".join(bad_tokens)
        + "\nFix scenario_badblueprint.toml in agentbeats-lambda branch `submission/purple_vanguard` to point to harness paths under `scenarios/security_arena/...`."
    )

must_exist = [
    harness_dir / "scenarios/security_arena/orchestrator.py",
    harness_dir / "scenarios/security_arena/agents/generic_attacker.py",
    harness_dir / "scenarios/security_arena/agents/generic_defender.py",
]
missing = [str(p) for p in must_exist if not p.exists()]
if missing:
    raise FileNotFoundError("Missing harness files:\n" + "\n".join(missing))
print("TOML + harness preflight guard passed.")


TOML + harness preflight guard passed.


## F. Install harness


In [27]:
import subprocess
from pathlib import Path

def sh(cmd: str):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Ensure /content exists (Colab-style)
Path("/content").mkdir(parents=True, exist_ok=True)

# Install Python 3.11 (Ubuntu)
sh("sudo apt-get update -y")
sh("sudo apt-get install -y python3.11 python3.11-venv python3.11-dev")

# Create harness venv
HARNESS_VENV = Path("/content/harness-venv")
if not HARNESS_VENV.exists():
    sh(f"python3.11 -m venv {HARNESS_VENV}")

HARNESS_PY = HARNESS_VENV / "bin" / "python"
HARNESS_PIP = HARNESS_VENV / "bin" / "pip"

# Upgrade pip tooling inside harness venv
sh(f"{HARNESS_PIP} install -U pip setuptools wheel")

print("Harness venv python:", HARNESS_PY)
print("Harness venv pip:", HARNESS_PIP)
sh(f"{HARNESS_PY} --version")

$ sudo apt-get update -y
Hit:1 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  InRelease
Ign:2 http://linux.mellanox.com/public/repo/doca/2.9.3/ubuntu22.04/x86_64 ./ InRelease
Hit:3 http://linux.mellanox.com/public/repo/doca/2.9.3/ubuntu22.04/x86_64 ./ Release
Hit:4 https://download.docker.com/linux/ubuntu jammy InRelease
Hit:5 https://packages.microsoft.com/repos/azure-cli jammy InRelease
Hit:7 http://archive.lambdalabs.com/ubuntu jammy InRelease
Hit:8 https://pkg.cloudflare.com/cloudflared jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:10 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:11 http://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:14 https://ppa.launchpadcontent.net/flexiondotorg/nvtop/ubuntu jammy InRelease
Reading package lists...
$ sudo apt-get install -y python3.11

In [29]:
import subprocess
import sys
from pathlib import Path

harness_dir = Path("/content/agentbeats-lambda")
if not harness_dir.exists():
    raise RuntimeError(f"Harness repo not found at {harness_dir}")

HARNESS_VENV = Path("/content/harness-venv")
HARNESS_PY = HARNESS_VENV / "bin" / "python"
HARNESS_PIP = HARNESS_VENV / "bin" / "pip"

if not HARNESS_PIP.exists():
    raise RuntimeError("Harness venv not found. Run the previous 'Create harness venv (Python 3.11+)' section first.")

# Install harness into the Python 3.11 venv (editable)
subprocess.run(f"{HARNESS_PIP} install -e {harness_dir}", shell=True, check=True, cwd=harness_dir)

# Detect CLI from harness pyproject scripts (prefer exact script name)
CLI_NAME = "agentbeats-run"

cli_path = HARNESS_VENV / "bin" / CLI_NAME
print("Detected harness CLI:", CLI_NAME)
print("Harness CLI path:", cli_path)

subprocess.run(f"{cli_path} --help", shell=True, check=True)


Obtaining file:///content/agentbeats-lambda
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for agentbeats-tutorial (pyproject.toml): started
  Building editable for agentbeats-tutorial (pyproject.toml): finished with status 'done'
  Created wheel for agentbeats-tutorial: filename=agentbeats_tutorial-0.1.0-py3-none-any.whl size=9917 sha256=01ac6b3798564f02e7a83f7b6658d44dc358e38b9aaa00a4a29dd6a4139e0f91
  Stored in directo

CompletedProcess(args='/content/harness-venv/bin/agentbeats-run --help', returncode=0)

## G. Configure harness to use local endpoint


In [30]:
import os
import re
from pathlib import Path

harness_dir = Path("/content/agentbeats-lambda")
if not harness_dir.exists():
    raise RuntimeError(f"Harness repo not found at {harness_dir}")

ENDPOINT = "http://127.0.0.1:8000/v1"

ENV_PATTERNS = [
    re.compile(r'os\.environ\[\s*"([A-Z0-9_]+)"\s*\]'),
    re.compile(r"os\.environ\[\s*'([A-Z0-9_]+)'\s*\]"),
    re.compile(r'getenv\(\s*"([A-Z0-9_]+)"\s*\)'),
    re.compile(r"getenv\(\s*'([A-Z0-9_]+)'\s*\)"),
]

def iter_text_files(root: Path):
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in {".py", ".toml", ".yaml", ".yml", ".md"}:
            yield p

env_names = set()
for p in iter_text_files(harness_dir):
    try:
        txt = p.read_text(errors="ignore")
    except Exception:
        continue
    for pat in ENV_PATTERNS:
        for m in pat.findall(txt):
            env_names.add(m)

if not env_names:
    raise RuntimeError("No environment variables found in harness source. Cannot configure endpoint reliably.")

endpoint_vars = [
    n for n in env_names
    if any(k in n for k in ["BASE_URL", "API_BASE", "ENDPOINT", "HOST", "URL"])
    and "MODEL" not in n
    and "KEY" not in n
]
key_vars = [n for n in env_names if "API_KEY" in n or n.endswith("_KEY")]
model_vars = [
    n for n in env_names
    if "MODEL" in n and not any(k in n for k in ["ENDPOINT", "BASE", "URL", "HOST"])
]

for n in endpoint_vars:
    os.environ[n] = ENDPOINT

for n in key_vars:
    os.environ.setdefault(n, "DUMMY_KEY")  # never print real secrets

for n in model_vars:
    os.environ.setdefault(n, "gpt-oss-20b")

os.environ.setdefault("MODEL_NAME", "gpt-oss-20b")

print("Configured endpoint:", ENDPOINT)
print("Set endpoint vars:", ", ".join(sorted(endpoint_vars)) if endpoint_vars else "(none found)")
print("Set key vars:", ", ".join(sorted(key_vars)) if key_vars else "(none found)")
print("Set model vars:", ", ".join(sorted(model_vars)) if model_vars else "(none found)")
print("MODEL_NAME:", os.environ.get("MODEL_NAME"))


Configured endpoint: http://127.0.0.1:8000/v1
Set endpoint vars: OPENAI_BASE_URL
Set key vars: OPENAI_API_KEY
Set model vars: (none found)
MODEL_NAME: gpt-oss-20b


## H. Run FULL SCORING


In [15]:
import os
import shutil
import subprocess
import time
from pathlib import Path

def run_full_scoring(timeout_sec: int | None = None):
    harness_dir = Path("/content/agentbeats-lambda")
    toml_path = harness_dir / "scenarios/security_arena/submissions/purple_vanguard/legaleagleeye/scenario_legaleagleeye.toml"

    if not harness_dir.exists():
        raise RuntimeError(f"Harness repo not found at {harness_dir}")
    if not toml_path.exists():
        raise RuntimeError(f"Scenario TOML not found at {toml_path}")

    # Preflight: ensure TOML no longer contains old paths
    toml_txt = toml_path.read_text()
    bad_tokens = []
    if "vendor/agentbeats-lambda/" in toml_txt:
        bad_tokens.append("vendor/agentbeats-lambda/")
    if bad_tokens:
        raise RuntimeError(
            "TOML still contains old paths: " + ", ".join(bad_tokens)
            + "\nFix scenario_badblueprint.toml in agentbeats-lambda branch `submission/purple_vanguard` "
            + "to point to harness paths under `scenarios/security_arena/...`."
        )

    # Preflight: ensure harness files exist
    must_exist = [
        harness_dir / "scenarios/security_arena/orchestrator.py",
        harness_dir / "scenarios/security_arena/agents/generic_attacker.py",
        harness_dir / "scenarios/security_arena/agents/generic_defender.py",
        harness_dir / "scenarios/security_arena/plugins/registry.py",
    ]
    missing = [str(p) for p in must_exist if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing harness files:\n" + "\n".join(missing))

    results_dir = Path("/content/results/mailmaninthemiddle")
    results_dir.mkdir(parents=True, exist_ok=True)
    log_path = results_dir / "full_score.log"

    # Use harness venv CLI (Python 3.11+)
    harness_venv = Path("/content/harness-venv")
    cli = harness_venv / "bin" / "agentbeats-run"
    if not cli.exists():
        raise RuntimeError(f"Harness CLI not found at {cli}. Run the 'Create harness venv' + 'Install harness' sections first.")

    # Optional: force OpenAI base URL/api key if your agents rely on env vars
    # (Keep these if your setup expects it; otherwise you can remove.)
    env = os.environ.copy()
    env.setdefault("OPENAI_BASE_URL", "http://127.0.0.1:8000/v1")
    env.setdefault("OPENAI_API_KEY", "local")

    cmd = [str(cli), "--show-logs", str(toml_path)]

    print("Running:", " ".join(cmd))
    print("Log ->", log_path)

    with open(log_path, "w", encoding="utf-8") as f:
        proc = subprocess.Popen(
            cmd,
            cwd=str(harness_dir),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        start = time.time()
        try:
            for line in proc.stdout:
                print(line, end="")
                f.write(line)
                if timeout_sec is not None and (time.time() - start) > timeout_sec:
                    raise TimeoutError(f"Scoring timed out after {timeout_sec}s. See log: {log_path}")
            rc = proc.wait(timeout=5)
        except TimeoutError:
            proc.kill()
            raise
        except Exception:
            proc.kill()
            raise

    if rc != 0:
        raise RuntimeError(f"Scoring failed (exit={rc}). See log: {log_path}")

    print("\n✅ FULL SCORING finished successfully.")
    print("Results directory:", results_dir)
    return log_path

run_full_scoring(timeout_sec=None)


Running: /content/harness-venv/bin/agentbeats-run --show-logs /content/agentbeats-lambda/scenarios/security_arena/submissions/purple_vanguard/legaleagleeye/scenario_legaleagleeye.toml
Log -> /content/results/mailmaninthemiddle/full_score.log
INFO:generic_orchestrator:Starting Security Arena Orchestrator on http://127.0.0.1:9010
INFO:     Started server process [19833]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:9010 (Press CTRL+C to quit)
INFO:     Started server process [19831]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:9021 (Press CTRL+C to quit)
INFO:     Started server process [19832]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:9020 (Press CTRL+C to quit)
Starting Generic Attacker on http://127.0.0.1:9021
Using OpenAI model: opena

PosixPath('/content/results/mailmaninthemiddle/full_score.log')

## I. Write score_status.json


In [44]:
import json
import shutil
from pathlib import Path

results_dir = Path("/content/results/badblueprint")
results_dir.mkdir(parents=True, exist_ok=True)
log_path = results_dir / "full_score.log"
status_path = results_dir / "score_status.json"

ok = log_path.exists() and log_path.stat().st_size > 0

# Copy agent-card*.json into results_dir (optional, best-effort)
candidates = list(Path("/content/agentbeats-lambda").rglob("agent-card*.json")) + list(Path("/content").rglob("agent-card*.json"))
seen = set()
for src in candidates:
    name = src.name
    if name in seen:
        continue
    try:
        shutil.copy2(src, results_dir / name)
        seen.add(name)
    except Exception:
        pass

payload = {
    "ok": bool(ok),
    "results_dir": str(results_dir),
    "log_path": str(log_path),
}
status_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote:", status_path)


Wrote: /content/results/badblueprint/score_status.json


## J. Package results for download


In [ ]:
import tarfile
from pathlib import Path

results_dir = Path("/content/results/badblueprint")
archive_path = Path("/content/results_badblueprint_colab.tgz")

with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(results_dir, arcname="badblueprint")

print("Archive created at:", archive_path)


## Cleanup (stop server)


In [ ]:
import os
import signal
import time
from pathlib import Path

SERVER_PID = Path("/content/transformers_server.pid")
if SERVER_PID.exists():
    pid = int(SERVER_PID.read_text().strip())
    try:
        os.killpg(pid, signal.SIGTERM)
        time.sleep(2)
        try:
            os.kill(pid, 0)
            os.killpg(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        print(f"Stopped server process group {pid}")
    except ProcessLookupError:
        print("Server process not running.")
    SERVER_PID.unlink(missing_ok=True)
else:
    print("No server PID file found.")

